In this series of examples, we will perform the uncertainty quantification procedure from start to finish. We will quantify the uncertainty in k_eff due to uncertainty in Pu-239's fission cross section data within a simplified model of the Jezebel experiment. In this notebook, we will focus on creating the necessary perturbed ACE files.

In [1]:
#Import necessary modules
import WINDIGO
import numpy as np
import os

#Set NJOY path
os.environ["NJOY"] = "/home/jacob/miniconda3/envs/openmc-env/bin/njoy"

We first need to generate an unperturbed ACE file for Pu-239. For the purposes of this analysis, we will use the energy bounds from the SCALE44 energy grid to define our perturbation boundaries. The SCALE44 energy grid is given here: https://serpent.vtt.fi/mediawiki/index.php/SCALE_44-group_structure

In [ ]:
# Input SCALE44 energy grid
scale44_grid_MeV = np.array([
    1.0000E-11, 3.0000E-9, 7.5000E-9, 1.0000E-8, 2.5300E-8,
    3.0000E-8, 4.0000E-8, 5.0000E-8, 7.0000E-8, 1.0000E-7,
    1.5000E-7, 2.0000E-7, 2.2500E-7, 2.5000E-7, 2.7500E-7,
    3.2500E-7, 3.5000E-7, 3.7500E-7, 4.0000E-7, 6.2500E-7,
    1.0000E-6, 1.7700E-6, 3.0000E-6, 4.7500E-6, 6.0000E-6,
    8.1000E-6, 1.0000E-5, 3.0000E-5, 1.0000E-4, 5.5000E-4,
    3.0000E-3, 1.7000E-2, 2.5000E-2, 1.0000E-1, 4.0000E-1,
    9.0000E-1, 1.4000E0, 1.8500E0, 2.3540E0, 2.4790E0,
    3.0000E0, 4.8000E0, 6.4340E0, 8.1873E0, 2.0000E1
])

# Convert to eV
scale44_grid_eV = scale44_grid_MeV * 10**6

# Generate the unperturbed Pu-239 ACE file
unperturbed_pu239_path = WINDIGO.generate_unperturbed_neutron_ace_file(
    frendy_Path="/mnt/c/Users/jacob/frendy_20241030",
    endf_Path=(
        "/mnt/c/Users/jacob/frendy_20241030/ENDF-B-VIII.0"
        "/ENDF-B-VIII.0/neutrons/n-094_Pu_239.endf"
    ),
    temperature=300,
    nuclide="Pu239",
    upgrade_Flag=True,
    energy_grid=scale44_grid_eV,
    cleanup_Flag=True
)

Next we will create our direct perturbation ACE files. We will apply a perturbation of +10% to the fission cross section.

In [ ]:
WINDIGO.generate_direct_perturbation_ace_files(
    frendy_Path="/mnt/c/Users/jacob/frendy_20241030",
    unperturbed_ACE_file_path=unperturbed_pu239_path,
    energy_grid=scale44_grid_MeV,
    mt_Number=18,
    nuclide="Pu239",
    perturbation_coefficient=1.10,
    cleanup_Flag=True
)

Intermediate Files Removed
All ACE files have successfully generated; they are located in: /mnt/c/Users/jacob/frendy_20241030/Pu239_DirectPerturbationACEFiles_ReactionMT_18


'/mnt/c/Users/jacob/frendy_20241030/Pu239_DirectPerturbationACEFiles_ReactionMT_18'

To generate our random sampling ACE files, we need to use Sandy to first retrieve a relative covariance matrix for Pu-239's fission cross section.

In [ ]:
result = WINDIGO.sandy_covariance_retrieval(
    energy_grid=scale44_grid_eV,
    nuclide="Pu239",
    mt_Number=18,
    data_library="endfb_80",
    temperature=300,
    relative_Flag=True,
    plotting_Flag=True
)

pu239_fission_relative_covariance_data_path = result[0]
pu239_fission_relative_covariance_plot_path = result[1]

With this covariance matrix, we can now generate our random sampling ACE files.

In [ ]:
WINDIGO.generate_random_sampling_ace_files(
    frendy_Path="/mnt/c/Users/jacob/frendy_20241030",
    unperturbed_ACE_file_path=unperturbed_pu239_path,
    relative_covariance_matrix_path=pu239_fission_relative_covariance_data_path,
    energy_grid=scale44_grid_MeV,
    mt_Number=18,
    nuclide="Pu239",
    sample_size=100,
    cleanup_Flag=True,
)

Perturbation factors created successfully
Intermediate Files Removed
All ACE files generated successfully and are located in: /mnt/c/Users/jacob/frendy_20241030/Pu239_RandomSamplingACEFiles_ReactionMT_18


'/mnt/c/Users/jacob/frendy_20241030/Pu239_RandomSamplingACEFiles_ReactionMT_18'

We now have all of the perturbed ACE files that we will need to perform our uncertainty quantification analysis.